# Exploration du référentiel des zones

**Objectif.** Comprendre `taxi_zone_lookup.csv`, le référentiel qui donne un nom et un arrondissement
à chaque code de zone (`PULocationID`, `DOLocationID`) du fichier de courses.

**Pourquoi c'est nécessaire.** Sans ce référentiel, on ne peut ni nommer les zones ni répondre à la
question métier « quelles zones génèrent le plus de revenus ? ». Il sert aussi à construire `dim_zone`
en couche Gold.

In [1]:
import pandas as pd

zones = pd.read_csv("../data/raw/taxi_zone_lookup.csv")
courses = pd.read_parquet(
    "../data/raw/yellow_tripdata_2024-01.parquet",
    columns=["PULocationID", "DOLocationID"],
)

*Note.* Le paramètre `columns` ne charge que les deux colonnes utiles. Parquet stocke les données
**par colonne** : on évite de lire les 17 autres, ce qui est plus rapide et consomme moins de mémoire.

## 1. Structure du référentiel

**Question.** Combien de zones, quelles colonnes, de quels types ?

In [2]:
print(zones.shape)
print(zones.dtypes)
zones.head(10)

(265, 4)
LocationID      int64
Borough           str
Zone              str
service_zone      str
dtype: object


,LocationID,Borough,Zone,service_zone
0,1,EWR,Newark Airport,EWR
1,2,Queens,Jamaica Bay,Boro Zone
2,3,Bronx,Allerton/Pelham Gardens,Boro Zone
3,4,Manhattan,Alphabet City,Yellow Zone
4,5,Staten Island,Arden Heights,Boro Zone
5,6,Staten Island,Arrochar/Fort Wadsworth,Boro Zone
6,7,Queens,Astoria,Boro Zone
7,8,Queens,Astoria Park,Boro Zone
8,9,Queens,Auburndale,Boro Zone
9,10,Queens,Baisley Park,Boro Zone


**Constat.**
- 265 zones, 4 colonnes : `LocationID` (entier), `Borough`, `Zone` et `service_zone` (texte).
- Chaque ligne décrit une zone de taxi : un code, son arrondissement, son nom et sa catégorie de service.

## 2. Qualité du référentiel

**Question.** Chaque `LocationID` est-il unique ? Quelles valeurs prend `Borough` ?
Y a-t-il des noms de zones en double ?

In [3]:
print("LocationID unique :", zones["LocationID"].is_unique)
print()
print(zones["Borough"].value_counts(dropna=False))
print()
zones[zones.duplicated("Zone", keep=False)].sort_values("Zone")

LocationID unique : True

Borough
Queens           69
Manhattan        69
Brooklyn         61
Bronx            43
Staten Island    20
EWR               1
Unknown           1
NaN               1
Name: count, dtype: int64



,LocationID,Borough,Zone,service_zone
55,56,Queens,Corona,Boro Zone
56,57,Queens,Corona,Boro Zone
102,103,Manhattan,Governor's Island/Ellis Island/Liberty Island,Yellow Zone
103,104,Manhattan,Governor's Island/Ellis Island/Liberty Island,Yellow Zone
104,105,Manhattan,Governor's Island/Ellis Island/Liberty Island,Yellow Zone


**Constat.**
- `LocationID` est unique : on peut s'en servir de clé pour `dim_zone`.
- `Borough` compte 5 vrais arrondissements (Queens 69, Manhattan 69, Brooklyn 61, Bronx 43, Staten Island 20) et `EWR` (l'aéroport de Newark, qui est au New Jersey, pas dans un arrondissement new-yorkais).
- Deux zones spéciales : 264 (`Borough = Unknown`, `Zone` et `service_zone` valent `N/A`) et 265 (`Borough = N/A`, `Zone = Outside of NYC`). Pandas lit le texte `N/A` comme une valeur manquante par défaut (`NaN`) ; avec `keep_default_na=False`, on le lit tel qu'il est écrit.
- Deux noms de zone apparaissent plusieurs fois avec des `LocationID` différents : `Corona` (56 et 57) et `Governor's Island/Ellis Island/Liberty Island` (103, 104 et 105).
- **Conséquence :** `dim_zone` doit être identifiée par `LocationID`, jamais par le nom, qui n'est pas unique.

## 3. Correspondance avec le fichier de courses

**Question.** Tous les codes de zone du fichier de courses existent-ils dans le référentiel ?
Un code absent ferait perdre des lignes lors de la jointure en couche Gold.

In [4]:
codes_ref = set(zones["LocationID"])
for col in ["PULocationID", "DOLocationID"]:
    inconnus = set(courses[col].unique()) - codes_ref
    n = courses[col].isin(inconnus).sum()
    print(f"{col} : {len(inconnus)} codes absents du référentiel, {n} lignes concernées")

PULocationID : 0 codes absents du référentiel, 0 lignes concernées
DOLocationID : 0 codes absents du référentiel, 0 lignes concernées


In [5]:
brut = pd.read_csv("../data/raw/taxi_zone_lookup.csv", keep_default_na=False)
brut[brut["LocationID"].isin([264, 265])]

,LocationID,Borough,Zone,service_zone
263,264,Unknown,N/A,N/A
264,265,N/A,Outside of NYC,N/A


In [6]:
for col in ["PULocationID", "DOLocationID"]:
    n = courses[col].isin([264, 265]).sum()
    print(f"{col} : {n:,} courses ({n / len(courses):.2%}) dans les zones 264 ou 265")

PULocationID : 12,018 courses (0.41%) dans les zones 264 ou 265
DOLocationID : 28,083 courses (0.95%) dans les zones 264 ou 265


**Constat.**
- Aucun code de zone du fichier de courses n'est absent du référentiel (0 sur `PULocationID`, 0 sur `DOLocationID`) : la jointure ne perdra aucune ligne.

## 4. Conclusions

- `dim_zone` utilise `LocationID` comme clé, jamais le nom de zone (`Corona` et `Governor's Island/Ellis Island/Liberty Island` ont plusieurs codes).
- Les zones 264 (« Unknown ») et 265 (« Outside of NYC ») sont de vraies zones, qui concernent 0,41 % des départs et 0,95 % des arrivées. Elles sont conservées dans `dim_zone` avec des libellés explicites, et on pourra les exclure des classements géographiques.
- Le texte `N/A` du CSV ne doit pas être converti en valeur manquante à l'ingestion : on lit le fichier avec `keep_default_na=False` pour conserver la source telle quelle (principe de la couche Bronze).
- La jointure entre les courses et le référentiel ne perd aucune ligne (0 code absent).